# Chapter 8 Exercise Solution - Phase Space in a Drift

A drift contains no magnetic or electric field, so the particle momentum remains constant and the particle travels along a straight line.

The reference particle travels along the reference direction. The tracked particle also has horizontal momentum, so it follows a diagonal line.

In [1]:
using SciBmad
using Printf

## First-principles calculation

The initial phase-space coordinates are

$$
(x,p_x,y,p_y,z,p_z)=(0,\;0.2,\;0,\;0,\;0,\;1).
$$

Since $p_z=(P-P_0)/P_0$, the normalized total momentum is

$$
\frac{P}{P_0}=1+p_z=2.
$$

The horizontal momentum is $P_x/P_0=p_x=0.2$. With no vertical momentum, the momentum along the reference direction is

$$
p_s=\frac{P_s}{P_0}=\sqrt{(1+p_z)^2-p_x^2}.
$$

The direction of the straight trajectory is therefore determined by

$$
\frac{dx}{ds}=\frac{p_x}{p_s}.
$$

After a drift of length $L$, the final horizontal position is

$$
x_f=L\frac{p_x}{p_s}.
$$

The diagonal path is longer than the reference particle's path. In the ultra-relativistic approximation, the particle therefore falls behind by

$$
z_f=L\left(1-\frac{1+p_z}{p_s}\right).
$$

In [2]:
L = 2.0
px = 0.2
pz = 1.0

p_total = 1 + pz
ps = sqrt(p_total^2 - px^2)
x_first_principles = L * px / ps
z_first_principles = L * (1 - p_total / ps)

@printf("P/P0  = %.12f\n", p_total)
@printf("Ps/P0 = %.12f\n", ps)
@printf("xf     = %.12f m\n", x_first_principles)
@printf("zf     = %.12f m\n", z_first_principles)

P/P0  = 2.000000000000
Ps/P0 = 1.989974874213
xf     = 0.201007563052 m
zf     = -0.010075630518 m


## SciBmad check

Now track the same initial particle through a $2\ \mathrm{m}$ drift in SciBmad and compare the final coordinates.

In [3]:
drift = Beamline(
    [Drift(L=L)];
    species_ref=Species("proton"),
    E_ref=1e12,
)

initial = [0.0, px, 0.0, 0.0, 0.0, pz]
particle = Bunch(copy(initial); species=drift.species_ref, p_over_q_ref=drift.p_over_q_ref)
track!(particle, drift)
final = copy(particle.coords.v[1, :])

@printf("SciBmad xf = %.12f m\n", final[1])
@printf("SciBmad zf = %.12f m\n", final[5])
@printf("|Delta x|  = %.3e m\n", abs(final[1] - x_first_principles))
@printf("|Delta z|  = %.3e m\n", abs(final[5] - z_first_principles))

@assert isapprox(final[1], x_first_principles; atol=1e-12)
@assert isapprox(final[5], z_first_principles; atol=1e-6)

SciBmad xf = 0.201007563052 m
SciBmad zf = -0.010074970252 m
|Delta x|  = 0.000e+00 m
|Delta z|  = 6.603e-07 m


The first-principles result and SciBmad tracking align. The small difference in $z$ comes from SciBmad retaining the finite proton mass, while the hand calculation assumes the particle speed is exactly the speed of light.